# Workflow Patterns

**Module:** 14 — AI Orchestration

Sequential, parallel, conditional, and event-driven patterns for GenAI systems.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Implement sequential, parallel, conditional, and event-driven flows
- Choose patterns using variability, latency, and consistency needs
- Combine patterns into a realistic ticket pipeline


## Pattern map

```mermaid
flowchart LR
  S[Sequential] --> P[Parallel]
  P --> C[Conditional]
  C --> E[Event-driven]
```

| Pattern | Sketch | Latency | Typical AI use |
|---------|--------|---------|----------------|
| Sequential | A→B→C | Sum of steps | Prompt chains, sanitize→generate→validate |
| Parallel | A∥B∥C→join | ~max(steps) | Multi-retriever, multi-critic |
| Conditional | if/else routes | Varies | Intent routing, escalation |
| Event-driven | on(event)→handler | Async | Webhooks, human approvals, cron |


## Sequential Pattern

### Definition
Steps run one after another; each consumes prior outputs.

### Why it matters
Simplest mental model; easy to test; often correct for dependent transforms.

### How it works
Pass a context object through pure-ish step functions; fail fast on errors; checkpoint between expensive steps.

### Intuition
An assembly line — station N needs station N-1's output.

### Pitfalls
- Hidden coupling via globals
- No partial resume
- Over-serializing independent work

### When to use
When step B truly needs step A's result.


In [ ]:
# Demo 1: sequential context pipeline
def normalize(ctx):
    ctx["text"] = ctx["text"].strip().lower()
    return ctx

def redact(ctx):
    ctx["text"] = ctx["text"].replace("ssn:1234", "[REDACTED]")
    return ctx

def classify(ctx):
    ctx["label"] = "sensitive" if "[redacted]" in ctx["text"] else "ok"
    return ctx

ctx = {"text": "  User SSN:1234  "}
for step in (normalize, redact, classify):
    ctx = step(ctx)
print(ctx)


## Parallel Pattern

### Definition
Independent steps execute concurrently, then join.

### Why it matters
Cuts latency when work does not depend on each other; common in RAG multi-query and ensemble critics.

### How it works
Fan-out tasks → gather results → merge with explicit strategy (concat, vote, weighted).

### Intuition
Several cooks prep ingredients at once, then plate together.

### Pitfalls
- Race conditions on shared mutable state
- Join timeouts
- Cost explosion from unbounded fan-out

### When to use
Independent retrievals, evaluations, or enrichments.


In [ ]:
# Demo 2: parallel retrievers + merge
from concurrent.futures import ThreadPoolExecutor, as_completed

def bm25(q): return [f"bm25:{q}"]
def vector(q): return [f"vec:{q}"]
def sql(q): return [f"sql:{q}"] if "id" in q else []

def parallel_retrieve(q):
    out = []
    with ThreadPoolExecutor(max_workers=3) as ex:
        futs = [ex.submit(fn, q) for fn in (bm25, vector, sql)]
        for f in as_completed(futs):
            out.extend(f.result())
    # dedupe preserve order
    seen, merged = set(), []
    for x in out:
        if x not in seen:
            seen.add(x); merged.append(x)
    return merged

print(parallel_retrieve("invoice id 42"))


## Conditional Pattern

### Definition
Branching control flow based on predicates (rules or model judgments).

### Why it matters
Most business AI needs routes: refund vs FAQ vs human.

### How it works
Compute a discriminant → select subgraph → optionally rejoin. Keep predicates testable.

### Intuition
A train switchyard — destination depends on the signal.

### Pitfalls
- Unstable LLM routers without fallback
- Missing else branch
- Nested spaghetti conditions

### When to use
Intent routing, threshold-based escalation, feature flags.


In [ ]:
# Demo 3: conditional router with fallback
def route(intent: str, confidence: float) -> str:
    if confidence < 0.55:
        return "human_review"
    return {
        "billing": "billing_flow",
        "tech": "tech_flow",
        "cancel": "retention_flow",
    }.get(intent, "faq_flow")

cases = [("billing", 0.9), ("billing", 0.4), ("unknown", 0.8)]
for intent, conf in cases:
    print(intent, conf, "->", route(intent, conf))


## Event-Driven Pattern

### Definition
Handlers react to events (webhooks, queues, approvals) rather than a single synchronous call stack.

### Why it matters
Humans, payment processors, and long jobs don't fit request/response alone.

### How it works
Emit domain events → durable queue → workers continue workflow with correlation/run ids.

### Intuition
Dominos: something happens, the next piece falls — possibly much later.

### Pitfalls
- Lost events without outbox pattern
- At-least-once delivery duplicates
- Missing correlation ids

### When to use
HITL, payment settlement, async document processing.


In [ ]:
# Demo 4: event bus + workflow continuation
from collections import defaultdict

class Bus:
    def __init__(self):
        self.handlers = defaultdict(list)
        self.dlq = []
    def on(self, event_type, fn):
        self.handlers[event_type].append(fn)
    def emit(self, event_type, payload):
        for fn in self.handlers.get(event_type, []):
            try:
                fn(payload)
            except Exception as e:
                self.dlq.append((event_type, payload, str(e)))

runs = {}
bus = Bus()

def on_approved(p):
    runs[p["run_id"]] = "resume_refund"

def on_paid(p):
    runs[p["run_id"]] = "send_receipt"

bus.on("approval.granted", on_approved)
bus.on("payment.ok", on_paid)
bus.emit("approval.granted", {"run_id": "r1"})
bus.emit("payment.ok", {"run_id": "r1"})
print(runs)


## Pattern Selection Guide

| If you need... | Prefer |
|----------------|--------|
| Deterministic compliance path | Sequential + conditional |
| Lower latency independent work | Parallel |
| Wait for humans/payments | Event-driven |
| Open-ended tool use | Agent loop *inside* a workflow boundary |

### Combined example — support ticket
1. Sequential: ingest → redact  
2. Parallel: retrieve KB ∥ fetch CRM ∥ sentiment  
3. Conditional: if risk high → human else auto-draft  
4. Event: on approval → send reply  


In [ ]:
# Demo 5: mini combined pipeline
def support_ticket(text: str, approved=False):
    redacted = text.replace("@", "[at]")
    kb, crm = [f"kb:{redacted[:12]}"], {"plan": "pro"}
    risk = "high" if "sue" in text.lower() else "low"
    if risk == "high" and not approved:
        return {"status": "awaiting_approval", "draft": None, "crm": crm}
    return {"status": "sent", "draft": f"Hello, re: {kb[0]}", "crm": crm}

print(support_ticket("I will sue you"))
print(support_ticket("I will sue you", approved=True))


### Try it yourself — Patterns

1. Add a join timeout simulation to parallel_retrieve.
2. Implement an else/default branch logger for unknown intents.
3. Model payment webhook retries (at-least-once) with idempotency keys.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `fan-out` | Start multiple parallel branches |
| `join` | Wait for branches to finish |
| `correlation id` | Links events to a run |
| `DLQ` | Dead-letter queue for failed events |


## Pattern Composition Catalog

| Composite | How |
|-----------|-----|
| Scatter-gather | Parallel + join merge |
| Router + specialist chains | Conditional → sequential |
| Saga | Sequential with compensations |
| Wait-for-signal | Event-driven + resume |
| Map-reduce over docs | Parallel map + sequential reduce |

### Saga sketch (refund)
`authorize → charge → provision` with compensations `void → refund → deprovision`


In [ ]:
# Saga with compensations
class Saga:
    def __init__(self):
        self.stack = []  # list of compensate callables
    def do(self, action, compensate):
        result = action()
        self.stack.append(compensate)
        return result
    def abort(self):
        while self.stack:
            self.stack.pop()()

log = []
s = Saga()
try:
    s.do(lambda: log.append("authorize") or "ok", lambda: log.append("void"))
    s.do(lambda: log.append("charge") or "ok", lambda: log.append("refund"))
    raise RuntimeError("provision failed")
except RuntimeError:
    s.abort()
print(log)


In [ ]:
# Event correlation
events = [
    {"type": "ticket.created", "run_id": "r1"},
    {"type": "approval.granted", "run_id": "r2"},
    {"type": "approval.granted", "run_id": "r1"},
]

def apply(events):
    state = {}
    for e in events:
        state.setdefault(e["run_id"], []).append(e["type"])
    return state

print(apply(events))


### Try it yourself — Patterns deepen

1. Implement join that fails if any branch raises.
2. Design a cron event that reopens stuck `waiting_human` > 24h.


## Key Takeaways

- Patterns are composable building blocks
- Parallelism needs merge strategies
- Events need durability and idempotency
- Select by dependency structure, not hype
